# 1) Creating synthetic data:
- Due to data sharing agreement, milk data cannot be disclosed. Likewise, weather data cannot be open because it is linked to test-day for each cow.
- This code generates the synthetic data to run the code.


# 1. import packages:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
import random
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
## making sure to stay under the project folder
root = Path.cwd().resolve().parents[0] 
sys.path.insert(0, str(root))
os.chdir(root)

# 2. generating synthetic data:

## 2-1. Generating synthetic data:
- In real data, we used energy-corrected milk which accounts for protein and fat (see the paper). Here we assume that "milk" is final energy corrected milk for simplicity.
- Given that PRISM and AgERA5 didn't have the data after June 2024, we limit our data to Jan 1st 2024 to Jun 30th, 2024.
- Conditions to make:
    - minimum 5 cows per herd
    - mininum 3 test-day records per parity
    - minimum 5 herds per county
    - variations between parities, counties, and national timeseries (see below)

In [ ]:
SEED = 125

random.seed(SEED)
np.random.seed(SEED)                      # seeds NumPy global RNG
rng = np.random.default_rng(SEED)

In [ ]:
# >5 counties per state: 2 states x 6 counties = 12 GEOIDs
STATE_CODES = ["48", "01"] # texas, wisconsin 
COUNTY_CODES = ["001", "002", "003", "004", "005", "006"]
GEOIDS = [s + c for s in STATE_CODES for c in COUNTY_CODES]  # 5-digit FIPS


In [ ]:
# Herd structure
HERDS_PER_COUNTY = 2
COWS_PER_HERD = 5  # >=5 cows per herd (exactly 5)

# DIM requirements
DIM_GRID = np.array([5, 15, 30, 45, 60, 80, 100, 120, 150, 180, 210, 240, 270, 305], dtype=int)
DIM_MIN_POINTS_PER_COW = 3

# Optional: per-cow dropout to introduce irregular sampling but preserve overall curve visibility
DIM_DROPOUT_RATE = 0.12  # each DIM point independently dropped with this probability
 

# Date bounds for test days (cow record dates)
DATE_START = pd.Timestamp("2000-01-01")
DATE_END = pd.Timestamp("2024-06-30")

# Parity restricted to 1-3
PARITY_CHOICES = np.array([1, 2, 3], dtype=int)
PARITY_PROBS = np.array([0.45, 0.35, 0.20])

# Birth before calving (synthetic)
BIRTH_DAYS_BEFORE_CALVING_MIN = 650
BIRTH_DAYS_BEFORE_CALVING_MAX = 1800

# Milk generation: parity effect + lactation-like DIM shape + noise
MILK_BASE_2000 = 28.0
NOISE_SD = 0.8  # keep modest so curve and trends are visible

# -----------------------------
# Herd-year coverage control
# Ensures per-herd quadratic increase is visible in cal_yr plots.
# -----------------------------
YEAR_GRID = np.array([2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024], dtype=int)

In [ ]:
herd_rows = []
for geoid in GEOIDS:
    for j in range(HERDS_PER_COUNTY):
        herd_rows.append({
            "GEOID": geoid,
            "herd_code": f"H_{geoid}_{j+1:02d}"
        })
herds = pd.DataFrame(herd_rows)


# County random effect (small shift) to make county averages differ
# deterministic by GEOID code for reproducibility
geoid_codes = herds["GEOID"].astype("category").cat.categories
geoid_to_effect = {}
for i, g in enumerate(geoid_codes):
    # small spread around 0
    geoid_to_effect[str(g)] = (i % 7 - 3) * 0.4  # approx -1.2 to +1.2


In [ ]:
# -----------------------------
# Assign herd-specific convex quadratic trend coefficients (c > 0)
# Use scaled time t in [0,1] where t = (cal_yr-2000)/24
# -----------------------------
coef = herds[["herd_code"]].drop_duplicates().sort_values("herd_code").reset_index(drop=True)
coef["a"] = rng.normal(0.0, 0.6, size=len(coef))                         # intercept
coef["b"] = rng.normal(1.2, 0.35, size=len(coef))                        # positive slope
coef["c"] = rng.lognormal(mean=np.log(2.8), sigma=0.35, size=len(coef))   # strictly positive curvature


In [ ]:

# -----------------------------
# Generate fully synthetic cow test-day rows
# -----------------------------
rows = []
cow_global_id = 0

for _, herd in herds.iterrows():
    geoid = herd["GEOID"]
    herd_code = herd["herd_code"]
    
    # Spread calving years across the range so the quadratic is visible.
    # Jitter the YEAR_GRID per herd a little so herds aren't identical.
    jitter = int(rng.integers(-1, 2))  # -1,0,1
    years_for_herd = np.clip(YEAR_GRID + jitter, 2000, 2024)

    # Allocate cows across years_for_herd (repeat if needed)
    cow_years = rng.choice(years_for_herd, size=COWS_PER_HERD, replace=True)

    for kcow in range(COWS_PER_HERD):
        cow_global_id += 1
        cow_id = f"COW{cow_global_id:07d}"

        parity = int(rng.choice(PARITY_CHOICES, p=PARITY_PROBS))

        # Choose calving_date within chosen year (random day-of-year)
        cal_yr = int(cow_years[kcow])
        doy = int(rng.integers(1, 366))
        calving_date = (pd.Timestamp(f"{cal_yr}-01-01") + pd.Timedelta(days=doy - 1)).normalize()

        # Ensure test-day dates stay within DATE_END given DIM max
        # If calving_date is too late in 2024, shift earlier.
        max_dim = int(DIM_GRID.max())
        if calving_date + pd.Timedelta(days=max_dim) > DATE_END:
            calving_date = (DATE_END - pd.Timedelta(days=max_dim)).normalize()

        # Birth date
        birth_days = int(rng.integers(BIRTH_DAYS_BEFORE_CALVING_MIN, BIRTH_DAYS_BEFORE_CALVING_MAX + 1))
        birth = (calving_date - pd.Timedelta(days=birth_days)).normalize()

        # DIM selection: start from common grid, drop some points
        keep = rng.random(len(DIM_GRID)) > DIM_DROPOUT_RATE
        dim = DIM_GRID[keep]
        if len(dim) < DIM_MIN_POINTS_PER_COW:
            dim = np.array([5, 100, 305], dtype=int)
        dim = np.unique(dim)
        dim.sort()

        # Record dates (merge key is GEOID+date)
        dates = pd.to_datetime([(calving_date + pd.Timedelta(days=int(d))).normalize() for d in dim])

        # -----------------------------
        # Lactation curve (normalized to [0,1]) to create a hump
        # -----------------------------
        dimf = dim.astype(float)
        rise = float(rng.uniform(10.0, 22.0))
        decay = float(rng.uniform(160.0, 240.0))
        lact = (1.0 - np.exp(-dimf / rise)) * np.exp(-dimf / decay)
        lact = lact / max(lact.max(), 1e-9)  # normalize

        # -----------------------------
        # National baseline trend (mild). You can set this to 0 if you only want herd trends.
        # This is based on calving year (not test-day year) to match your cal_yr plots.
        # -----------------------------
        national = MILK_BASE_2000 + 0.15 * (cal_yr - 2000)  # mild upward drift

        # -----------------------------
        # Parity-specific magnitude + shape:
        # - parity 1: lower level + flatter curve
        # -----------------------------
        if parity == 1:
            parity_level = -2.8
            lact_floor = 0.82
            hump_strength = 0.30
        elif parity == 2:
            parity_level = 0.0
            lact_floor = 0.62
            hump_strength = 0.85
        else:  # parity == 3
            parity_level = 1.8
            lact_floor = 0.55
            hump_strength = 1.00

        lact_mult = lact_floor + (1.0 - lact_floor) * (hump_strength * lact)

        # County effect
        county_effect = geoid_to_effect[geoid]

        # Base milk before herd trend/noise
        noise = rng.normal(0.0, NOISE_SD, size=len(dim))
        milk = (national + parity_level + county_effect) * lact_mult + noise
        milk = np.clip(milk, 0.0, None)

        # Assemble rows (milk trend added later, once herd coefs are merged)
        df = pd.DataFrame({
            "GEOID": geoid,
            "herd_code": herd_code,
            "cow_id": cow_id,
            "birth": birth,
            "calving_date": calving_date,
            "parity": parity,
            "days_in_milk": dim.astype(int),
            "date": dates,
            "cal_yr": cal_yr,
            "milk": milk
        })

        rows.append(df)
out = pd.concat(rows, ignore_index=True)
out['cal_yr'] = out['date'].dt.year

In [ ]:
out[['GEOID','herd_code','cow_id','date']][out[['GEOID','herd_code','cow_id','date']].duplicated()]

In [ ]:
# Dates within range and increasing within cow
if out["date"].min() < DATE_START or out["date"].max() > DATE_END:
    raise RuntimeError("Date constraint violated: dates out of 2000–2024 range.")

## 2-2. Visual inspection:

In [ ]:
## national level:
fig, axs = plt.subplots(figsize=(8,3), nrows=1, ncols=2, tight_layout=True)
ax= axs.flatten()
out.groupby(['days_in_milk','parity'])['milk'].mean().unstack().plot(ax=ax[0], marker='o', markersize=2, linestyle='')
ax[0].set_title('milk per parity')
out.groupby(['cal_yr'])['milk'].mean().plot(ax=ax[1])
ax[1].set_title('milk averages over time')
plt.show()

In [ ]:
## county and herd level:
fig, axs = plt.subplots(figsize=(8,3), nrows=1, ncols=2, tight_layout=True)
ax = axs.flatten()
out.groupby(['days_in_milk','GEOID'])['milk'].mean().unstack().plot(ax=ax[0],marker='o',linestyle='',markersize=2)
ax[0].legend('')
ax[0].set_title('county-specific milk across days in milk')

out.groupby(['cal_yr','herd_code'])['milk'].mean().unstack().plot(ax=ax[1], marker='o',linestyle='', markersize=2)
ax[1].legend('')
ax[1].set_title('herd-specific yearly milk')

In [ ]:
for geoid in out['herd_code'].unique():
    fig, ax = plt.subplots(figsize=[4,3])
    out.loc[out['herd_code'] == geoid].groupby(['cal_yr'])['milk'].mean().plot(ax=ax) #marker='o', linestyle='',markersize=2,ax=ax)
    fig.show()

# 3. Weather data processing: 

In [ ]:
## Load weather:
## reading weather data
weather_data = '/projects/kyoung2@colostate.edu/dairy_newest/1_data/1_climate_data/3_weather_3days_avg_df_final_update.gzip'

w_lin_avg = ['GEOID','date','w_lin_avg_tmax', 'w_lin_avg_tmin', 'w_lin_avg_tdmean',
       'w_lin_avg_vpdmax', 'w_lin_avg_vpdmin', 'w_lin_avg_ppt',
       'w_lin_avg_tmean', 'w_lin_avg_rh', 'w_lin_avg_rh_pm', 'w_lin_avg_rh_am',
       'w_lin_avg_wetbT', 'w_lin_avg_thi_max',
       'w_lin_avg_thi_min', 'w_lin_avg_thi_avg', 'w_lin_avg_adjthi',
       'w_lin_avg_ag_tmax', 'w_lin_avg_ag_tmin', 'w_lin_avg_ag_tmean',
       'w_lin_avg_ag_tdmean', 'w_lin_avg_ag_rh_am', 'w_lin_avg_ag_rh_pm',
       'w_lin_avg_ag_ppt', 'w_lin_avg_ag_cloud', 'w_lin_avg_ag_ssrd_wm-2',
       'w_lin_avg_ag_wind_2m', 'w_lin_avg_ag_wetbT', 
       'w_lin_avg_ag_thi_max', 'w_lin_avg_ag_thi_min', 'w_lin_avg_ag_adjthi']

weather = pd.read_parquet(weather_data, columns=w_lin_avg)
weather['state_code'] = weather['GEOID'].str[:2]
weather = weather.rename(columns=lambda c: c.replace("w_lin_avg_",""))

In [ ]:
out['year_month'] = out['date'].dt.strftime('%Y-%m')
weather['year_month'] = weather['date'].dt.strftime('%Y-%m')
out = out.rename(columns={'date':'fake_date'})

In [ ]:

## assigning the weather data from warm and cool states:
temp = weather.loc[weather['state_code'].isin(['26','27','55','06','40','48','12','01'])].copy()
candidates = temp['GEOID'].unique()
count_st = {'48':[], '26':[], '27':[],'55':[], '06':[], '12':[],'01':[],'40':[]}
mapping = {}
for sg, g_out in out.groupby("GEOID", sort=False):
    needed = set(g_out["year_month"].values)
    chosen = None
    
    for rg in candidates:
        dates_rg = temp.loc[temp["GEOID"] == rg, "year_month"].values
        if needed.issubset(set(dates_rg)):
            chosen = rg
            count_st[rg[:2]].append(rg)
            candidates = candidates[candidates != rg]
            
            if len(count_st['55']) + len(count_st['27']) +len(count_st['26']) >5:
                candidates = candidates[~np.char.startswith(candidates.astype(str), '55')]
                candidates = candidates[~np.char.startswith(candidates.astype(str), '27')]
                candidates = candidates[~np.char.startswith(candidates.astype(str), '26')]
            
            if len(count_st[rg[:2]]) > 5:
                candidates = candidates[~np.char.startswith(candidates.astype(str), rg[:2])]
            break

    if chosen is None:
        raise RuntimeError(f"No candidate county found for synthetic GEOID {sg}.")

    mapping[sg] = chosen

In [ ]:
mapping

In [ ]:
temp = temp.rename(columns={'GEOID':'true_geoid'})
out['true_geoid'] = out['GEOID'].map(mapping)

In [ ]:
out.loc[out['true_geoid'].str[:2] == '06']['true_geoid'].unique()

In [ ]:
del weather
temp.loc[(~temp['true_geoid'].isin(out.loc[out['true_geoid'].str[:2] == '06']['true_geoid'].unique())) & (temp['state_code'] == '06')]['true_geoid'].unique()

In [ ]:
## merging:
final_out = out.merge(temp.drop_duplicates(['true_geoid','year_month']), on=['true_geoid','year_month'], how='left')

In [ ]:
## cleaning:
final_out = final_out.rename(columns={'GEOID':'fake_geoid'})
final_out['GEOID'] = final_out['true_geoid']
final_out.loc[final_out['true_geoid'] == '12067','GEOID'] = '06029'
final_out.loc[final_out['true_geoid'] == '12121','GEOID'] = '06107'

In [ ]:
final_out = final_out.drop(columns=['fake_date','fake_geoid','true_geoid'])

In [ ]:
## combining with county centroid:
counties = pd.read_csv('1_data/US_county_boundary.csv', index_col=0)
counties['GEOID'] = counties['GEOID'].astype(int).astype(str).str.zfill(5)
counties = counties[['STATE','GEOID','lon','lat']].rename(columns={'STATE':'state_abv'})

final_out = counties.merge(final_out, on=['GEOID'], how='right')

## saving:
final_out.drop(columns=['state_code']).to_parquet('1_data/final_synthetic_df.gzip', compression='gzip')